# Preparación de datos para modelado predictivo de demanda

Este notebook desarrolla la preparación del dataset consolidado para la evaluación de algoritmos de forecast aplicados a artículos previamente filtrados como aptos para predicción.

La base de entrada corresponde al dataframe consolidado posterior a la etapa de clusterización, donde cada artículo conserva su historial de ventas y adicionalmente cuenta con una asignación de cluster, una descripción interpretativa del tipo de demanda y una lista preliminar de modelos candidatos de forecast.

## Contexto metodológico

En los notebooks anteriores se realizó primero una fotografía matemática del comportamiento histórico de ventas de los artículos aplicables a forecast. Posteriormente, se ejecutó un proceso de clusterización utilizando variables asociadas a volumen, frecuencia, recencia, variabilidad, concentración de picos, tendencia y posible estacionalidad.

El resultado de la clusterización permitió segmentar los artículos en grupos matemáticamente diferenciados. Estos clusters no representan todavía modelos predictivos, sino perfiles de comportamiento de demanda que servirán como punto de partida para seleccionar y comparar familias de algoritmos de forecast.

## Objetivo del notebook

El objetivo principal de este notebook es transformar el dataset consolidado en una estructura adecuada para entrenamiento y evaluación de modelos predictivos. Para ello, se preparará una base supervisada donde cada observación represente el comportamiento histórico de un artículo en un período determinado y se construyan variables explicativas a partir de información pasada.

La finalidad es evitar data leakage y garantizar que los modelos utilicen únicamente información disponible antes del período que se desea predecir.

## Procedimiento general

1. Importar el dataframe consolidado con historial de ventas, cluster y modelos candidatos.
2. Validar la estructura del dataset y las columnas necesarias.
3. Identificar las columnas mensuales de venta histórica.
4. Convertir el dataset de formato ancho a formato largo, generando una estructura artículo-mes.
5. Ordenar cronológicamente las ventas por artículo.
6. Construir variables temporales tipo lag.
7. Construir variables móviles, como medias, desviaciones y ventas acumuladas.
8. Definir la variable objetivo de predicción.
9. Integrar variables de clusterización y atributos maestros del artículo.
10. Preparar divisiones temporales de entrenamiento y prueba.
11. Definir métricas de evaluación para comparar algoritmos.
12. Preparar la base final para el entrenamiento de modelos por cluster.

## Enfoque de predicción

El primer enfoque de modelado será supervisado. En lugar de predecir directamente desde una única fila por artículo, se construirá una tabla histórica en formato artículo-mes. Esta estructura permitirá generar variables basadas en períodos anteriores, como ventas rezagadas, promedios móviles y señales recientes de demanda.

La variable objetivo inicial será la demanda del siguiente período mensual. Posteriormente podrá evaluarse un segundo objetivo basado en demanda acumulada en una ventana futura, por ejemplo tres meses, con el fin de acercar el modelo a decisiones de reabastecimiento.

## Relación con la clusterización

La columna `cluster_kmeans_final` será utilizada como una variable metodológica clave. Esta permitirá evaluar si diferentes grupos de artículos presentan mejores resultados con distintas familias de modelos.

Los modelos candidatos asignados en el capítulo anterior no representan todavía el modelo ganador. Su función es orientar la matriz inicial de experimentación. La selección definitiva se realizará empíricamente mediante métricas de error y validación temporal.

## Consideraciones importantes

Durante esta etapa será fundamental evitar el uso de información futura para construir variables predictoras. Todas las variables utilizadas para predecir un mes determinado deberán calcularse únicamente con información disponible en meses anteriores.

Además, debido a la naturaleza de los repuestos industriales, se prestará especial atención a artículos con demanda intermitente, valores en cero, picos de venta y diferencias significativas entre clusters.

In [30]:
# ============================================================
# 1. Importación de librerías
# ============================================================
# En esta sección se importan las librerías necesarias para:
# - manipulación de datos
# - preparación temporal del dataset
# - construcción de variables supervisadas
# - validación de estructura
# ============================================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from pathlib import Path

In [31]:
# ============================================================
# 2. Carga del dataframe consolidado
# ============================================================
# Este archivo corresponde al dataset histórico enriquecido con:
# - cluster_kmeans_final
# - descripcion_cluster
# - modelos_candidatos_forecast
# - tipo_demanda
# - modelo_forecast preliminar
# ============================================================

ruta_archivo = "./Datos/dataframe_aplica_forecast_true_clusterizado_modelos_2026_07_18.xlsx"

df_modelado = pd.read_excel(
    ruta_archivo
)

print("Dimensiones del dataframe consolidado:")
print(df_modelado.shape)

df_modelado.head()

Dimensiones del dataframe consolidado:
(450, 128)


,codigo_articulo,descripcion_completa,grupo_articulo,categoria,subcategoria,tipo,unidad_medida_inventario,marca_fabricante,ultima_fecha_compra,ultimo_precio_compra,...,monto_riesgo_inmediato,monto_riesgo_proximo,score_prioridad_articulo,venta_mensual_en_revision,venta_promedio_reciente_last_6m,ratio_venta_reciente_6m,motivo_aplica_forecast,cluster_kmeans_final,descripcion_cluster,modelos_candidatos_forecast
0,US2: QF120A,Interruptor 20 A 1P 120 V 10K QPF2 GFCI 5MA,EQUI.AUTO.Y CONTROL,EQUIPO ELECTRICO & AUTOMATIZACION,CENTROS DE CARGA E INTERRUPTORES,NaN,UNI,SIEMENS,2026-06-26,32.680000,...,0.000000,46283.713583,46283.713583,20634.356250,395.500000,0.845612,Aplica forecast,2.0,Demanda activa y reciente con señales de creci...,"Exponential Smoothing, ARIMA/AutoARIMA, Random..."
1,W8,"Cinta para husos, Habasit, W-8",FAJAS,BANDAS,CORREAS PLANAS DE TRANSMISION,CORREAS PLANAS DE TRANSMISION,CM2,HABASIT ROCUA,2026-01-31,0.007366,...,0.000000,0.000000,0.000000,0.000000,240040.970000,0.964050,Aplica forecast,1.0,"Demanda históricamente relevante, variable y c...","ETS con tendencia amortiguada, ARIMA/AutoARIMA..."
2,RB330-3PLY-30''-SHRD,"Banda de Hule y lona de 30""",FAJAS,CORREAS,REDONDAS,REDONDAS,MT,SHARDA,2026-02-27,19.335359,...,0.000000,3384.525384,3384.525384,0.000000,43.551667,0.507788,Aplica forecast,1.0,"Demanda históricamente relevante, variable y c...","ETS con tendencia amortiguada, ARIMA/AutoARIMA..."
3,C3.5-TEH-14.984,RODILLO RETRO PSV/1.20F14.89N.38,POLEAS RODI.Y TAMB.,NaN,NaN,NaN,UNI,PRECISION INC,2026-05-31,51.088667,...,0.000000,0.000000,0.000000,0.000000,32.166667,1.349650,Aplica forecast,2.0,Demanda activa y reciente con señales de creci...,"Exponential Smoothing, ARIMA/AutoARIMA, Random..."
4,RB330-3PLY-24''-SHRD,"Banda de Hule y lona de 24"" 3 lonas",FAJAS,CORREAS,REDONDAS,REDONDAS,MT,SHARDA,2026-06-01,21.100484,...,7043.938685,0.000000,7043.938685,1654.161893,71.051667,1.223552,Aplica forecast,2.0,Demanda activa y reciente con señales de creci...,"Exponential Smoothing, ARIMA/AutoARIMA, Random..."


In [32]:
# ============================================================
# 3. Validación inicial del dataframe consolidado
# ============================================================
# Objetivo:
# Confirmar que el dataframe contiene las columnas necesarias
# para iniciar la preparación del dataset supervisado de forecast.
# ============================================================

print("Dimensiones del dataframe:")
print(df_modelado.shape)

print("\nColumnas disponibles:")
print(df_modelado.columns.tolist())

print("\nTipos de datos:")
df_modelado.info()

Dimensiones del dataframe:
(450, 128)

Columnas disponibles:
['codigo_articulo', 'descripcion_completa', 'grupo_articulo', 'categoria', 'subcategoria', 'tipo', 'unidad_medida_inventario', 'marca_fabricante', 'ultima_fecha_compra', 'ultimo_precio_compra', 'costo_promedio', 'codigo_bodega', 'stock_en_mano', 'stock_comprometido', 'stock_en_pedido_transito', 'total_unidades_vendidas_24m', 'venta_anual_promedio_24m', 'conteo_facturas_24m', 'cantidad_clientes_24m', 'frecuencia_venta_mensual_24m', 'conteo_compras_proveedores_24m', 'cantidad_proveedores_24m', 'frecuencia_compra_mensual_24m', 'meses_promedio_entre_compras_24m', 'cantidad_entradas_compra_con_oc', 'lead_time_promedio_dias', 'lead_time_min_dias', 'lead_time_max_dias', 'ultima_entrada_mercancia', 'cantidad_promedio_pedido_24m', 'cliente_1', 'cliente_2', 'cliente_3', 'cliente_4', 'cliente_5', 'cantidad_minima_compra', 'stock_minimo', 'stock_maximo', 'precio_lista_11_promociones_ecommerce', 'precio_lista_12_b2b_vip', 'precio_lista_13

In [33]:
# ============================================================
# 4. Identificación de columnas mensuales de venta
# ============================================================
# Objetivo:
# Detectar únicamente las columnas mensuales de venta.
# Se excluyen columnas agregadas como venta_total_2024,
# venta_total_2025 y venta_total_2026.
# ============================================================

meses_es = [
    "enero", "febrero", "marzo", "abril", "mayo", "junio",
    "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"
]

columnas_venta_mensual = [
    col for col in df_modelado.columns
    if col.startswith("venta_")
    and any(mes in col for mes in meses_es)
]

print("Cantidad de columnas mensuales detectadas:")
print(len(columnas_venta_mensual))

print("\nColumnas mensuales detectadas:")
columnas_venta_mensual

Cantidad de columnas mensuales detectadas:
31

Columnas mensuales detectadas:


['venta_enero_2024',
 'venta_febrero_2024',
 'venta_marzo_2024',
 'venta_abril_2024',
 'venta_mayo_2024',
 'venta_junio_2024',
 'venta_julio_2024',
 'venta_agosto_2024',
 'venta_septiembre_2024',
 'venta_octubre_2024',
 'venta_noviembre_2024',
 'venta_diciembre_2024',
 'venta_enero_2025',
 'venta_febrero_2025',
 'venta_marzo_2025',
 'venta_abril_2025',
 'venta_mayo_2025',
 'venta_junio_2025',
 'venta_julio_2025',
 'venta_agosto_2025',
 'venta_septiembre_2025',
 'venta_octubre_2025',
 'venta_noviembre_2025',
 'venta_diciembre_2025',
 'venta_enero_2026',
 'venta_febrero_2026',
 'venta_marzo_2026',
 'venta_abril_2026',
 'venta_mayo_2026',
 'venta_junio_2026',
 'venta_julio_2026']

In [34]:
# ============================================================
# 5. Validar exclusión de columnas agregadas
# ============================================================

columnas_totales_detectadas = [
    col for col in columnas_venta_mensual
    if "total" in col
]

if columnas_totales_detectadas:
    print("⚠️ Se detectaron columnas totales dentro de columnas_venta_mensual:")
    print(columnas_totales_detectadas)
else:
    print("✅ No se incluyeron columnas agregadas de venta total.")

✅ No se incluyeron columnas agregadas de venta total.


In [35]:
# ============================================================
# 6. Ordenar columnas mensuales cronológicamente
# ============================================================

mapa_meses = {
    "enero": 1,
    "febrero": 2,
    "marzo": 3,
    "abril": 4,
    "mayo": 5,
    "junio": 6,
    "julio": 7,
    "agosto": 8,
    "septiembre": 9,
    "octubre": 10,
    "noviembre": 11,
    "diciembre": 12
}


def extraer_fecha_columna_venta(nombre_columna):
    """
    Convierte una columna tipo venta_enero_2024 en fecha 2024-01-28.
    """
    partes = nombre_columna.replace("venta_", "").split("_")
    
    mes_texto = partes[0]
    anio = int(partes[1])
    mes = mapa_meses[mes_texto]

    fecha_inicio_mes = pd.Timestamp(
        year=anio,
        month=mes,
        day=1
    )
    
    fecha_fin_mes = fecha_inicio_mes + pd.offsets.MonthEnd(0)
    
    return fecha_fin_mes 


columnas_venta_mensual = sorted(
    columnas_venta_mensual,
    key=extraer_fecha_columna_venta
)

print("Columnas mensuales ordenadas cronológicamente:")
columnas_venta_mensual

Columnas mensuales ordenadas cronológicamente:


['venta_enero_2024',
 'venta_febrero_2024',
 'venta_marzo_2024',
 'venta_abril_2024',
 'venta_mayo_2024',
 'venta_junio_2024',
 'venta_julio_2024',
 'venta_agosto_2024',
 'venta_septiembre_2024',
 'venta_octubre_2024',
 'venta_noviembre_2024',
 'venta_diciembre_2024',
 'venta_enero_2025',
 'venta_febrero_2025',
 'venta_marzo_2025',
 'venta_abril_2025',
 'venta_mayo_2025',
 'venta_junio_2025',
 'venta_julio_2025',
 'venta_agosto_2025',
 'venta_septiembre_2025',
 'venta_octubre_2025',
 'venta_noviembre_2025',
 'venta_diciembre_2025',
 'venta_enero_2026',
 'venta_febrero_2026',
 'venta_marzo_2026',
 'venta_abril_2026',
 'venta_mayo_2026',
 'venta_junio_2026',
 'venta_julio_2026']

In [36]:
# ============================================================
# 7. Validar rango temporal de ventas
# ============================================================

fechas_ventas = [
    extraer_fecha_columna_venta(col)
    for col in columnas_venta_mensual
]

print("Primera fecha de venta detectada:")
print(min(fechas_ventas))

print("\nÚltima fecha de venta detectada:")
print(max(fechas_ventas))

print("\nCantidad de meses detectados:")
print(len(fechas_ventas))

Primera fecha de venta detectada:
2024-01-31 00:00:00

Última fecha de venta detectada:
2026-07-31 00:00:00

Cantidad de meses detectados:
31


In [38]:
# ============================================================
# 8. Validar columnas clave para modelado
# ============================================================

columnas_clave_modelado = [
    "codigo_articulo",
    "descripcion_completa",
    "grupo_articulo",
    "categoria",
    "subcategoria",
    "tipo",
    "marca_fabricante",
    "unidad_medida_inventario",
    "cluster_kmeans_final",
    "descripcion_cluster",
    "modelos_candidatos_forecast",
    "tipo_demanda",
    "modelo_forecast",
    "lead_time_promedio_dias",
    "lead_time_min_dias",
    "lead_time_max_dias",
    "vendedor_1",
    "participacion_vendedor_1_24m",
    "punto_stock_reorden",
    "venta_promedio_mensual_24m",
    "alcance_inventario_24m",
    "meses_objetivo_cobertura",
    "deficit_punto_reorden",
    "decision_compra",
    "score_prioridad_articulo",

]

columnas_faltantes_modelado = [
    col for col in columnas_clave_modelado
    if col not in df_modelado.columns
]

if columnas_faltantes_modelado:
    print("⚠️ Columnas clave faltantes:")
    print(columnas_faltantes_modelado)
else:
    print("✅ Todas las columnas clave de modelado están disponibles.")

✅ Todas las columnas clave de modelado están disponibles.


In [39]:
# ============================================================
# 9. Validar y convertir ventas mensuales a numérico
# ============================================================

df_modelado[columnas_venta_mensual] = (
    df_modelado[columnas_venta_mensual]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
)

resumen_ventas_mensuales = (
    df_modelado[columnas_venta_mensual]
    .describe()
    .T
    .round(2)
)
resumen_ventas_mensuales

,count,mean,std,min,25%,50%,75%,max
venta_enero_2024,450.0,5339.93,65059.03,-5.0,0.0,0.0,10.00,1135971.33
venta_febrero_2024,450.0,2271.21,17497.97,0.0,0.0,0.0,5.75,216021.75
venta_marzo_2024,450.0,2916.36,31115.72,0.0,0.0,0.0,4.75,556403.94
venta_abril_2024,450.0,5898.09,42693.65,-16.8,0.0,0.0,6.00,637875.00
venta_mayo_2024,450.0,3567.06,25768.23,0.0,0.0,0.0,6.00,299707.00
venta_junio_2024,450.0,2251.99,19214.56,-3.0,0.0,0.0,4.75,315024.19
venta_julio_2024,450.0,2012.21,16190.06,0.0,0.0,0.0,8.00,269213.16
venta_agosto_2024,450.0,6043.96,59022.69,-18994.5,0.0,0.0,5.00,835958.03
venta_septiembre_2024,450.0,7905.01,73345.14,0.0,0.0,1.0,6.00,1347209.33
venta_octubre_2024,450.0,3866.75,32250.08,0.0,0.0,1.0,6.00,550345.95


In [40]:
# 10. Validar ventas negativas
# ============================================================
# Una venta negativa puede representar devolución, nota de crédito
# o ajuste. Para forecast de demanda normalmente debe revisarse.
# ============================================================

ventas_negativas = (df_modelado[columnas_venta_mensual] < 0).sum().sum()

print(f"Cantidad total de celdas mensuales con venta negativa: {ventas_negativas}")

if ventas_negativas > 0:
    columnas_con_negativos = (
        (df_modelado[columnas_venta_mensual] < 0)
        .sum()
        .sort_values(ascending=False)
    )

    display(columnas_con_negativos[columnas_con_negativos > 0])

Cantidad total de celdas mensuales con venta negativa: 28


venta_mayo_2026          8
venta_junio_2026         2
venta_octubre_2025       2
venta_septiembre_2025    2
venta_abril_2026         2
venta_abril_2025         2
venta_enero_2024         1
venta_junio_2025         1
venta_agosto_2025        1
venta_junio_2024         1
venta_abril_2024         1
venta_diciembre_2024     1
venta_febrero_2025       1
venta_noviembre_2024     1
venta_agosto_2024        1
venta_enero_2026         1
dtype: int64

In [42]:

# 11. Tratamiento de ventas negativas
# ============================================================
# Objetivo:
# Convertir a cero cualquier venta mensual negativa.
#
# Justificación:
# Para fines de forecast de demanda, los valores negativos no
# representan demanda real, sino posibles devoluciones, notas de crédito
# o ajustes contables. Por ello, se sustituyen por cero antes de construir
# las series temporales por artículo.
# ============================================================

# Conteo de valores negativos antes del tratamiento
ventas_negativas_antes = (df_modelado[columnas_venta_mensual] < 0).sum().sum()

print(f"Cantidad de celdas mensuales con venta negativa antes del ajuste: {ventas_negativas_antes}")

# Reemplazar ventas negativas por cero
df_modelado[columnas_venta_mensual] = df_modelado[columnas_venta_mensual].clip(lower=0)

# Conteo de valores negativos después del tratamiento
ventas_negativas_despues = (df_modelado[columnas_venta_mensual] < 0).sum().sum()

print(f"Cantidad de celdas mensuales con venta negativa después del ajuste: {ventas_negativas_despues}")

Cantidad de celdas mensuales con venta negativa antes del ajuste: 28
Cantidad de celdas mensuales con venta negativa después del ajuste: 0


In [45]:
# ============================================================
# 11.2 Validación estadística posterior al ajuste
# ============================================================

resumen_ventas_mensuales_post_ajuste = (
    df_modelado[columnas_venta_mensual]
    .describe()
    .T
    .round(2)
)

resumen_ventas_mensuales_post_ajuste

,count,mean,std,min,25%,50%,75%,max
venta_enero_2024,450.0,5339.94,65059.03,0.0,0.0,0.0,10.00,1135971.33
venta_febrero_2024,450.0,2271.21,17497.97,0.0,0.0,0.0,5.75,216021.75
venta_marzo_2024,450.0,2916.36,31115.72,0.0,0.0,0.0,4.75,556403.94
venta_abril_2024,450.0,5898.13,42693.64,0.0,0.0,0.0,6.00,637875.00
venta_mayo_2024,450.0,3567.06,25768.23,0.0,0.0,0.0,6.00,299707.00
venta_junio_2024,450.0,2251.99,19214.56,0.0,0.0,0.0,4.75,315024.19
venta_julio_2024,450.0,2012.21,16190.06,0.0,0.0,0.0,8.00,269213.16
venta_agosto_2024,450.0,6086.17,59011.53,0.0,0.0,0.0,5.00,835958.03
venta_septiembre_2024,450.0,7905.01,73345.14,0.0,0.0,1.0,6.00,1347209.33
venta_octubre_2024,450.0,3866.75,32250.08,0.0,0.0,1.0,6.00,550345.95


In [50]:
# ============================================================
# 12. Preparación de columnas base para formato largo
# ============================================================
# Objetivo:
# Definir qué columnas se mantendrán como atributos del artículo
# al convertir el dataframe de formato ancho a formato largo.
# ============================================================

columnas_id_modelado = [
    "codigo_articulo",
    "descripcion_completa",
    "grupo_articulo",
    "categoria",
    "subcategoria",
    "tipo",
    "unidad_medida_inventario",
    "marca_fabricante",
    "cluster_kmeans_final",
    "descripcion_cluster",
    "modelos_candidatos_forecast",
    "modelo_forecast",
    "tipo_demanda",
    "lead_time_promedio_dias",
    "stock_en_mano",
    "stock_comprometido",
    "stock_en_pedido_transito",
    "costo_promedio",
    "precio_referencia",
    "criticidad_comercial"
]

# Mantener únicamente columnas existentes
columnas_id_modelado = [
    col for col in columnas_id_modelado
    if col in df_modelado.columns
]

print("Columnas de identificación y contexto seleccionadas:")
print(columnas_id_modelado)
print(len(columnas_id_modelado ))

Columnas de identificación y contexto seleccionadas:
['codigo_articulo', 'descripcion_completa', 'grupo_articulo', 'categoria', 'subcategoria', 'tipo', 'unidad_medida_inventario', 'marca_fabricante', 'cluster_kmeans_final', 'descripcion_cluster', 'modelos_candidatos_forecast', 'modelo_forecast', 'tipo_demanda', 'lead_time_promedio_dias', 'stock_en_mano', 'stock_comprometido', 'stock_en_pedido_transito', 'costo_promedio', 'precio_referencia', 'criticidad_comercial']
20


In [48]:
# ============================================================
# 13. Conversión de ventas mensuales a formato largo
# ============================================================
# Objetivo:
# Convertir las columnas mensuales de venta en una estructura
# artículo-mes, necesaria para construir series temporales
# individuales por artículo.
# ============================================================

df_series = df_modelado.melt(
    id_vars=columnas_id_modelado,
    value_vars=columnas_venta_mensual,
    var_name="columna_venta",
    value_name="venta_mensual"
)

print("Dimensiones del dataframe en formato largo:")
print(df_series.shape)

df_series.head()

Dimensiones del dataframe en formato largo:
(13950, 22)


,codigo_articulo,descripcion_completa,grupo_articulo,categoria,subcategoria,tipo,unidad_medida_inventario,marca_fabricante,cluster_kmeans_final,descripcion_cluster,...,tipo_demanda,lead_time_promedio_dias,stock_en_mano,stock_comprometido,stock_en_pedido_transito,costo_promedio,precio_referencia,criticidad_comercial,columna_venta,venta_mensual
0,US2: QF120A,Interruptor 20 A 1P 120 V 10K QPF2 GFCI 5MA,EQUI.AUTO.Y CONTROL,EQUIPO ELECTRICO & AUTOMATIZACION,CENTROS DE CARGA E INTERRUPTORES,NaN,UNI,SIEMENS,2.0,Demanda activa y reciente con señales de creci...,...,Demanda activa y reciente con señales de creci...,28.633333,202.00,0.0,1300,0,44.118000,alta,venta_enero_2024,282.00
1,W8,"Cinta para husos, Habasit, W-8",FAJAS,BANDAS,CORREAS PLANAS DE TRANSMISION,CORREAS PLANAS DE TRANSMISION,CM2,HABASIT ROCUA,1.0,"Demanda históricamente relevante, variable y c...",...,"Demanda históricamente relevante, variable y c...",55.000000,997713.14,0.0,1270500,0,0.009944,alta,venta_enero_2024,760551.23
2,RB330-3PLY-30''-SHRD,"Banda de Hule y lona de 30""",FAJAS,CORREAS,REDONDAS,REDONDAS,MT,SHARDA,1.0,"Demanda históricamente relevante, variable y c...",...,"Demanda históricamente relevante, variable y c...",110.800000,573.06,0.0,0,0,26.102735,alta,venta_enero_2024,20.55
3,C3.5-TEH-14.984,RODILLO RETRO PSV/1.20F14.89N.38,POLEAS RODI.Y TAMB.,NaN,NaN,NaN,UNI,PRECISION INC,2.0,Demanda activa y reciente con señales de creci...,...,Demanda activa y reciente con señales de creci...,37.000000,202.00,5.0,0,0,68.969700,alta,venta_enero_2024,0.00
4,RB330-3PLY-24''-SHRD,"Banda de Hule y lona de 24"" 3 lonas",FAJAS,CORREAS,REDONDAS,REDONDAS,MT,SHARDA,2.0,Demanda activa y reciente con señales de creci...,...,Demanda activa y reciente con señales de creci...,120.333333,73.38,58.7,0,0,28.485653,alta,venta_enero_2024,76.56


In [49]:
# ============================================================
# 14. Crear columna fecha_mes
# ============================================================
# Objetivo:
# Convertir columnas como venta_enero_2024 en fechas reales
# correspondientes al último día de cada mes.
# ============================================================

df_series["fecha_mes"] = df_series["columna_venta"].apply(
    extraer_fecha_columna_venta
)

df_series[["codigo_articulo", "columna_venta", "fecha_mes", "venta_mensual"]].head()

,codigo_articulo,columna_venta,fecha_mes,venta_mensual
0,US2: QF120A,venta_enero_2024,2024-01-31,282.00
1,W8,venta_enero_2024,2024-01-31,760551.23
2,RB330-3PLY-30''-SHRD,venta_enero_2024,2024-01-31,20.55
3,C3.5-TEH-14.984,venta_enero_2024,2024-01-31,0.00
4,RB330-3PLY-24''-SHRD,venta_enero_2024,2024-01-31,76.56


In [52]:
# ============================================================
# 15. Ordenamiento cronológico por artículo
# ============================================================

df_series = df_series.sort_values(
    by=["codigo_articulo", "fecha_mes"]
).reset_index(drop=True)

df_series[
    [
        "codigo_articulo",
        "fecha_mes",
        "venta_mensual",
        "cluster_kmeans_final",
        "descripcion_cluster",
        "modelos_candidatos_forecast"
    ]
].head(20)

,codigo_articulo,fecha_mes,venta_mensual,cluster_kmeans_final,descripcion_cluster,modelos_candidatos_forecast
0,0225-3M,2024-01-31,0.0,1.0,"Demanda históricamente relevante, variable y c...","ETS con tendencia amortiguada, ARIMA/AutoARIMA..."
1,0225-3M,2024-02-29,0.0,1.0,"Demanda históricamente relevante, variable y c...","ETS con tendencia amortiguada, ARIMA/AutoARIMA..."
2,0225-3M,2024-03-31,0.0,1.0,"Demanda históricamente relevante, variable y c...","ETS con tendencia amortiguada, ARIMA/AutoARIMA..."
3,0225-3M,2024-04-30,0.0,1.0,"Demanda históricamente relevante, variable y c...","ETS con tendencia amortiguada, ARIMA/AutoARIMA..."
4,0225-3M,2024-05-31,0.0,1.0,"Demanda históricamente relevante, variable y c...","ETS con tendencia amortiguada, ARIMA/AutoARIMA..."
5,0225-3M,2024-06-30,0.0,1.0,"Demanda históricamente relevante, variable y c...","ETS con tendencia amortiguada, ARIMA/AutoARIMA..."
6,0225-3M,2024-07-31,0.0,1.0,"Demanda históricamente relevante, variable y c...","ETS con tendencia amortiguada, ARIMA/AutoARIMA..."
7,0225-3M,2024-08-31,18.0,1.0,"Demanda históricamente relevante, variable y c...","ETS con tendencia amortiguada, ARIMA/AutoARIMA..."
8,0225-3M,2024-09-30,0.0,1.0,"Demanda históricamente relevante, variable y c...","ETS con tendencia amortiguada, ARIMA/AutoARIMA..."
9,0225-3M,2024-10-31,0.0,1.0,"Demanda históricamente relevante, variable y c...","ETS con tendencia amortiguada, ARIMA/AutoARIMA..."


In [55]:
# ============================================================
# 16. Validación de longitud de series por artículo
# ============================================================

longitud_series = (
    df_series
    .groupby("codigo_articulo")
    .agg(
        meses_disponibles=("fecha_mes", "nunique"),
        venta_total_serie=("venta_mensual", "sum"),
        meses_con_venta=("venta_mensual", lambda x: (x > 0).sum())
    )
    .reset_index()
)

longitud_series.describe().round(2)

,meses_disponibles,venta_total_serie,meses_con_venta
count,450.0,450.00,450.00
mean,31.0,108213.38,14.88
std,0.0,741570.67,6.45
min,31.0,13.00,6.00
25%,31.0,37.25,10.00
50%,31.0,92.00,13.00
75%,31.0,447.50,19.00
max,31.0,8048540.06,31.00


In [56]:
# ============================================================
# 16.1 Validar artículos con meses incompletos
# ============================================================

articulos_meses_incompletos = longitud_series[
    longitud_series["meses_disponibles"] != len(columnas_venta_mensual)
]

print("Artículos con meses incompletos:")
print(articulos_meses_incompletos.shape[0])

articulos_meses_incompletos.head()

Artículos con meses incompletos:
0


,codigo_articulo,meses_disponibles,venta_total_serie,meses_con_venta


In [57]:
# ============================================================
# 17. Creación de variables temporales
# ============================================================
# Objetivo:
# Agregar variables de calendario que puedan servir como features
# para modelos supervisados.
# ============================================================

df_series["anio"] = df_series["fecha_mes"].dt.year
df_series["mes"] = df_series["fecha_mes"].dt.month
df_series["trimestre"] = df_series["fecha_mes"].dt.quarter

df_series[
    [
        "codigo_articulo",
        "fecha_mes",
        "anio",
        "mes",
        "trimestre",
        "venta_mensual"
    ]
].head(10)

,codigo_articulo,fecha_mes,anio,mes,trimestre,venta_mensual
0,0225-3M,2024-01-31,2024,1,1,0.0
1,0225-3M,2024-02-29,2024,2,1,0.0
2,0225-3M,2024-03-31,2024,3,1,0.0
3,0225-3M,2024-04-30,2024,4,2,0.0
4,0225-3M,2024-05-31,2024,5,2,0.0
5,0225-3M,2024-06-30,2024,6,2,0.0
6,0225-3M,2024-07-31,2024,7,3,0.0
7,0225-3M,2024-08-31,2024,8,3,18.0
8,0225-3M,2024-09-30,2024,9,3,0.0
9,0225-3M,2024-10-31,2024,10,4,0.0


In [58]:
# ============================================================
# 18. Creación de variables lag por artículo
# ============================================================
# Objetivo:
# Crear variables que representen ventas pasadas.
#
# venta_lag_1  = venta del mes anterior
# venta_lag_2  = venta de hace dos meses
# venta_lag_3  = venta de hace tres meses
# venta_lag_6  = venta de hace seis meses
# venta_lag_12 = venta del mismo mes del año anterior
# venta_lag_24 = venta del mismo mes dos años antes
# ============================================================

lags = [1, 2, 3, 6, 12, 24]

for lag in lags:
    df_series[f"venta_lag_{lag}"] = (
        df_series
        .groupby("codigo_articulo")["venta_mensual"]
        .shift(lag)
    )

df_series[
    [
        "codigo_articulo",
        "fecha_mes",
        "venta_mensual",
        "venta_lag_1",
        "venta_lag_2",
        "venta_lag_3",
        "venta_lag_6",
        "venta_lag_12",
        "venta_lag_24"
    ]
].head(30)

,codigo_articulo,fecha_mes,venta_mensual,venta_lag_1,venta_lag_2,venta_lag_3,venta_lag_6,venta_lag_12,venta_lag_24
0,0225-3M,2024-01-31,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,0225-3M,2024-02-29,0.0,0.0,NaN,NaN,NaN,NaN,NaN
2,0225-3M,2024-03-31,0.0,0.0,0.0,NaN,NaN,NaN,NaN
3,0225-3M,2024-04-30,0.0,0.0,0.0,0.0,NaN,NaN,NaN
4,0225-3M,2024-05-31,0.0,0.0,0.0,0.0,NaN,NaN,NaN
5,0225-3M,2024-06-30,0.0,0.0,0.0,0.0,NaN,NaN,NaN
6,0225-3M,2024-07-31,0.0,0.0,0.0,0.0,0.0,NaN,NaN
7,0225-3M,2024-08-31,18.0,0.0,0.0,0.0,0.0,NaN,NaN
8,0225-3M,2024-09-30,0.0,18.0,0.0,0.0,0.0,NaN,NaN
9,0225-3M,2024-10-31,0.0,0.0,18.0,0.0,0.0,NaN,NaN


In [60]:
# ============================================================
# 19. Creación de medias móviles por artículo
# ============================================================
# Objetivo:
# Crear predictores simples basados en promedios históricos.
#
# Se usa shift(1) para evitar data leakage.
# Es decir, para calcular la media móvil de un mes, solo se usan
# meses anteriores.
# ============================================================

ventanas_medias_moviles = [3, 6, 12, 24]

for ventana in ventanas_medias_moviles:
    df_series[f"media_movil_{ventana}m"] = (
        df_series
        .groupby("codigo_articulo")["venta_mensual"]
        .transform(
            lambda x: x.shift(1).rolling(
                window=ventana,
                min_periods=1
            ).mean().round(2)
        )
    )

df_series[
    [
        "codigo_articulo",
        "fecha_mes",
        "venta_mensual",
        "media_movil_3m",
        "media_movil_6m",
        "media_movil_12m",
        "media_movil_24m"
    ]
].head(30)

,codigo_articulo,fecha_mes,venta_mensual,media_movil_3m,media_movil_6m,media_movil_12m,media_movil_24m
0,0225-3M,2024-01-31,0.0,NaN,NaN,NaN,NaN
1,0225-3M,2024-02-29,0.0,0.00,0.00,0.00,0.00
2,0225-3M,2024-03-31,0.0,0.00,0.00,0.00,0.00
3,0225-3M,2024-04-30,0.0,0.00,0.00,0.00,0.00
4,0225-3M,2024-05-31,0.0,0.00,0.00,0.00,0.00
5,0225-3M,2024-06-30,0.0,0.00,0.00,0.00,0.00
6,0225-3M,2024-07-31,0.0,0.00,0.00,0.00,0.00
7,0225-3M,2024-08-31,18.0,0.00,0.00,0.00,0.00
8,0225-3M,2024-09-30,0.0,6.00,3.00,2.25,2.25
9,0225-3M,2024-10-31,0.0,6.00,3.00,2.00,2.00


In [61]:
# ============================================================
# 20. Definición de train/test split temporal
# ============================================================
# Objetivo:
# Separar los datos respetando la secuencia temporal.
#
# Se reservan los últimos 3 meses como test.
# Esto permitirá comparar los modelos baseline contra los modelos
# avanzados bajo el mismo período de evaluación.
# ============================================================

n_meses_test = 3

fechas_unicas = (
    df_series["fecha_mes"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

fechas_test = fechas_unicas.tail(n_meses_test)

fecha_inicio_test = fechas_test.min()
fecha_fin_test = fechas_test.max()

print("Cantidad total de meses disponibles:")
print(len(fechas_unicas))

print("\nRango completo:")
print(fechas_unicas.min(), "→", fechas_unicas.max())

print("\nFechas seleccionadas para test:")
print(fechas_test.tolist())

print("\nInicio test:")
print(fecha_inicio_test)

print("\nFin test:")
print(fecha_fin_test)

Cantidad total de meses disponibles:
31

Rango completo:
2024-01-31 00:00:00 → 2026-07-31 00:00:00

Fechas seleccionadas para test:
[Timestamp('2026-05-31 00:00:00'), Timestamp('2026-06-30 00:00:00'), Timestamp('2026-07-31 00:00:00')]

Inicio test:
2026-05-31 00:00:00

Fin test:
2026-07-31 00:00:00


In [62]:
# ============================================================
# 21. Crear datasets de train y test
# ============================================================

df_train = df_series[
    df_series["fecha_mes"] < fecha_inicio_test
].copy()

df_test = df_series[
    df_series["fecha_mes"].isin(fechas_test)
].copy()

print("Rango temporal train:")
print(df_train["fecha_mes"].min(), "→", df_train["fecha_mes"].max())

print("\nRango temporal test:")
print(df_test["fecha_mes"].min(), "→", df_test["fecha_mes"].max())

print("\nDimensiones train:")
print(df_train.shape)

print("\nDimensiones test:")
print(df_test.shape)

print("\nArtículos únicos train:")
print(df_train["codigo_articulo"].nunique())

print("\nArtículos únicos test:")
print(df_test["codigo_articulo"].nunique())

Rango temporal train:
2024-01-31 00:00:00 → 2026-04-30 00:00:00

Rango temporal test:
2026-05-31 00:00:00 → 2026-07-31 00:00:00

Dimensiones train:
(12600, 36)

Dimensiones test:
(1350, 36)

Artículos únicos train:
450

Artículos únicos test:
450


In [64]:
df_series

,codigo_articulo,descripcion_completa,grupo_articulo,categoria,subcategoria,tipo,unidad_medida_inventario,marca_fabricante,cluster_kmeans_final,descripcion_cluster,...,venta_lag_1,venta_lag_2,venta_lag_3,venta_lag_6,venta_lag_12,venta_lag_24,media_movil_3m,media_movil_6m,media_movil_12m,media_movil_24m
0,0225-3M,"Manga de sincronismo, 225-3M, Longitud externa...",FAJAS,CORREAS,SINCRONISMO,DIENTE CURVILINEO,MM,JASON,1.0,"Demanda históricamente relevante, variable y c...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0225-3M,"Manga de sincronismo, 225-3M, Longitud externa...",FAJAS,CORREAS,SINCRONISMO,DIENTE CURVILINEO,MM,JASON,1.0,"Demanda históricamente relevante, variable y c...",...,0.0,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00
2,0225-3M,"Manga de sincronismo, 225-3M, Longitud externa...",FAJAS,CORREAS,SINCRONISMO,DIENTE CURVILINEO,MM,JASON,1.0,"Demanda históricamente relevante, variable y c...",...,0.0,0.0,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00
3,0225-3M,"Manga de sincronismo, 225-3M, Longitud externa...",FAJAS,CORREAS,SINCRONISMO,DIENTE CURVILINEO,MM,JASON,1.0,"Demanda históricamente relevante, variable y c...",...,0.0,0.0,0.0,NaN,NaN,NaN,0.00,0.00,0.00,0.00
4,0225-3M,"Manga de sincronismo, 225-3M, Longitud externa...",FAJAS,CORREAS,SINCRONISMO,DIENTE CURVILINEO,MM,JASON,1.0,"Demanda históricamente relevante, variable y c...",...,0.0,0.0,0.0,NaN,NaN,NaN,0.00,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13945,XPZ1400,"Correa trapezoidal metrica, XPZ1400, Longitud ...",FAJAS,CORREAS,TRAPEZOIDALES,METRICAS,UNI,BANDO,0.0,Demanda con picos frecuentes y variabilidad co...,...,8.0,5.0,10.0,0.0,0.0,2.0,7.67,4.67,3.08,4.25
13946,XPZ1400,"Correa trapezoidal metrica, XPZ1400, Longitud ...",FAJAS,CORREAS,TRAPEZOIDALES,METRICAS,UNI,BANDO,0.0,Demanda con picos frecuentes y variabilidad co...,...,0.0,8.0,5.0,5.0,0.0,0.0,4.33,4.67,3.08,4.17
13947,XPZ1400,"Correa trapezoidal metrica, XPZ1400, Longitud ...",FAJAS,CORREAS,TRAPEZOIDALES,METRICAS,UNI,BANDO,0.0,Demanda con picos frecuentes y variabilidad co...,...,0.0,0.0,8.0,0.0,6.0,0.0,2.67,3.83,3.08,4.17
13948,XPZ1400,"Correa trapezoidal metrica, XPZ1400, Longitud ...",FAJAS,CORREAS,TRAPEZOIDALES,METRICAS,UNI,BANDO,0.0,Demanda con picos frecuentes y variabilidad co...,...,0.0,0.0,0.0,10.0,3.0,0.0,0.00,3.83,2.58,4.17


In [65]:
# ============================================================
# 22. Generación de inferencias baseline
# ============================================================
# Objetivo:
# Crear predicciones simples para los meses de test.
#
# pred_naive_1m:
#   predice usando la venta del mes anterior.
#
# pred_ma_3m_1m:
#   predice usando el promedio de los últimos 3 meses.
#
# pred_ma_6m_1m:
#   predice usando el promedio de los últimos 6 meses.
#
# pred_ma_12m_1m:
#   predice usando el promedio de los últimos 12 meses.
#
# pred_ma_24m_1m:
#   predice usando el promedio de los últimos 24 meses.
# ============================================================

df_series["pred_naive_1m"] = df_series["venta_lag_1"]
df_series["pred_ma_3m_1m"] = df_series["media_movil_3m"]
df_series["pred_ma_6m_1m"] = df_series["media_movil_6m"]
df_series["pred_ma_12m_1m"] = df_series["media_movil_12m"]
df_series["pred_ma_24m_1m"] = df_series["media_movil_24m"]

# Reconstruir df_test para incluir las nuevas predicciones
df_test = df_series[
    df_series["fecha_mes"].isin(fechas_test)
].copy()

df_test[
    [
        "codigo_articulo",
        "fecha_mes",
        "venta_mensual",
        "pred_naive_1m",
        "pred_ma_3m_1m",
        "pred_ma_6m_1m",
        "pred_ma_12m_1m",
        "pred_ma_24m_1m",
        "cluster_kmeans_final",
        "descripcion_cluster"
    ]
].head(20)

,codigo_articulo,fecha_mes,venta_mensual,pred_naive_1m,pred_ma_3m_1m,pred_ma_6m_1m,pred_ma_12m_1m,pred_ma_24m_1m,cluster_kmeans_final,descripcion_cluster
28,0225-3M,2026-05-31,24.0,0.0,8.00,6.00,5.50,9.75,1.0,"Demanda históricamente relevante, variable y c..."
29,0225-3M,2026-06-30,9.0,24.0,8.00,8.00,6.67,10.75,1.0,"Demanda históricamente relevante, variable y c..."
30,0225-3M,2026-07-31,0.0,9.0,11.00,9.50,7.42,11.12,1.0,"Demanda históricamente relevante, variable y c..."
59,0270H,2026-05-31,0.0,19.0,85.67,205.33,168.33,132.17,2.0,Demanda activa y reciente con señales de creci...
60,0270H,2026-06-30,300.0,0.0,73.00,105.33,135.00,131.75,2.0,Demanda activa y reciente con señales de creci...
61,0270H,2026-07-31,350.0,300.0,106.33,117.83,160.00,144.25,2.0,Demanda activa y reciente con señales de creci...
90,0384-3M-JASON,2026-05-31,0.0,0.0,0.00,21.67,15.83,12.29,0.0,Demanda con picos frecuentes y variabilidad co...
91,0384-3M-JASON,2026-06-30,0.0,0.0,0.00,16.67,14.58,12.29,0.0,Demanda con picos frecuentes y variabilidad co...
92,0384-3M-JASON,2026-07-31,0.0,0.0,0.00,7.50,14.58,12.29,0.0,Demanda con picos frecuentes y variabilidad co...
121,0390H-JASON,2026-05-31,25.0,201.0,75.33,37.67,44.67,29.62,2.0,Demanda activa y reciente con señales de creci...


In [73]:
# ============================================================
# 23. Validación manual de un artículo
# ============================================================
# Objetivo:
# Revisar visualmente si las predicciones baseline se calculan
# correctamente usando información anterior al mes evaluado.
# ============================================================

codigo_ejemplo = df_test["codigo_articulo"].iloc[0]

df_series[
    df_series["codigo_articulo"] == codigo_ejemplo
][
    [
        "codigo_articulo",
        "fecha_mes",
        "venta_mensual",
        "venta_lag_1",
        "media_movil_3m",
        "media_movil_6m",
        "media_movil_12m",
        "media_movil_24m",
        "pred_naive_1m",
        "pred_ma_3m_1m",
        "pred_ma_6m_1m",
        "pred_ma_12m_1m",
        "pred_ma_24m_1m"
    ]
].tail(13)

,codigo_articulo,fecha_mes,venta_mensual,venta_lag_1,media_movil_3m,media_movil_6m,media_movil_12m,media_movil_24m,pred_naive_1m,pred_ma_3m_1m,pred_ma_6m_1m,pred_ma_12m_1m,pred_ma_24m_1m
18,0225-3M,2025-07-31,14.0,0.0,3.33,26.67,14.83,9.89,0.0,3.33,26.67,14.83,9.89
19,0225-3M,2025-08-31,0.0,14.0,8.00,29.00,16.00,10.11,14.0,8.00,29.00,16.00,10.11
20,0225-3M,2025-09-30,0.0,0.0,4.67,4.00,14.50,9.60,0.0,4.67,4.00,14.50,9.60
21,0225-3M,2025-10-31,6.0,0.0,4.67,4.00,14.50,9.14,0.0,4.67,4.00,14.50,9.14
22,0225-3M,2025-11-30,12.0,6.0,2.00,5.00,15.00,9.00,6.0,2.00,5.00,15.00,9.00
23,0225-3M,2025-12-31,0.0,12.0,6.00,5.33,16.00,9.13,12.0,6.00,5.33,16.00,9.13
24,0225-3M,2026-01-31,0.0,0.0,6.00,5.33,16.00,8.75,0.0,6.00,5.33,16.00,8.75
25,0225-3M,2026-02-28,24.0,0.0,4.00,3.00,16.00,8.75,0.0,4.00,3.00,16.00,8.75
26,0225-3M,2026-03-31,0.0,24.0,8.00,7.00,5.50,9.75,24.0,8.00,7.00,5.50,9.75
27,0225-3M,2026-04-30,0.0,0.0,8.00,7.00,5.50,9.75,0.0,8.00,7.00,5.50,9.75


In [69]:
(df_series["codigo_articulo"] == codigo_ejemplo)["venta_mensual"].tail(12).sum()

KeyError: 'venta_mensual'